# Reannotación HaMeR -- S2 a S99

Pipeline completo, resumible. Por cada sujeto:
1. Descarga zip de Dropbox a `/content/`
2. Extrae frames a `/content/`
3. Corre MediaPipe + HaMeR local
4. Guarda CSV en Drive (append, flush por frame)
5. Borra zip y frames de `/content/`

Si la sesión muere: ejecuta Run All, continúa desde donde quedó.

**Requisitos en Drive:**
- `hamer_ckpt/_DATA/hamer_ckpts/checkpoints/hamer.safetensors`
- `hamer_ckpt/_DATA/data/mano/MANO_RIGHT.pkl`
- `hograspnet_hamer/hograspnet_mano.csv` (el CSV completo de 3.1 GB)

In [ ]:
# Celda 1: Instalar dependencias
!pip install -q fastapi uvicorn nest-asyncio
!pip install -q git+https://github.com/Schmetzler/hamer_keypoints.git
!pip install -q "mediapipe==0.10.13"

In [ ]:
# Celda 2: Montar Drive + cargar HaMeR
from google.colab import drive
drive.mount('/content/drive')

import torch, cv2, numpy as np, inspect
inspect.getargspec = inspect.getfullargspec
import hamer.configs as hcfg

HAMER_DATA_PATH = "/content/drive/MyDrive/hamer_ckpt/_DATA"
hcfg.CACHE_DIR_HAMER = HAMER_DATA_PATH

from hamer.hamer_module import HAMER
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

hamer_model = HAMER(mode="IMAGE", device=device)
dummy = np.zeros((224, 224, 3), dtype=np.uint8)
hamer_model.process(dummy, {"Right": [0.0, 0.0, 1.0, 1.0]}, rescale_factor=1.0)
print("HaMeR OK")

In [ ]:
# Celda 3: Configuracion
from pathlib import Path

# Drive
DRIVE_DIR      = Path("/content/drive/MyDrive/AIST-hand/data/raw")
CSV_IN         = DRIVE_DIR / "hograspnet_mano.csv"
CSV_OUT        = DRIVE_DIR / "hograspnet_hamer_all.csv"
LOG_FAILED     = DRIVE_DIR / "hograspnet_hamer_failed.txt"
PROGRESS_FILE  = DRIVE_DIR / "hograspnet_hamer_progress.txt"

# Local (efimero)
CONTENT_DIR    = Path("/content")

# Sujetos a procesar (S1 ya esta hecho, empezamos en S2)
START_SUBJECT  = 2
END_SUBJECT    = 99  # inclusivo

# URLs de Dropbox (S1=indice 0, S2=indice 1, ...)
SOURCE_URLS = [
    "https://www.dropbox.com/scl/fi/v31b80uns0v05emjoehjz/GraspNet_S1_1_Source_data.zip?rlkey=hrkswuxufbby01qq4zqjfy0bq&dl=1",
    "https://www.dropbox.com/scl/fi/526mamb8nogb6gkbpmysv/GraspNet_S2_1_Source_data.zip?rlkey=cvygh7qev4amh52l01rz9ad1r&dl=1",
    "https://www.dropbox.com/scl/fi/deh4ga22v83qx698kmc6a/GraspNet_S3_1_Source_data.zip?rlkey=67v6tx3muncheqgb2zr7x3hyp&dl=1",
    "https://www.dropbox.com/scl/fi/itcv902cn6ag6fhfw7z3a/GraspNet_S4_1_Source_data.zip?rlkey=ty8z9xlmkbivfumfcqyiz890y&dl=1",
    "https://www.dropbox.com/scl/fi/cu2qyafhyxvqquwdsksds/GraspNet_S5_1_Source_data.zip?rlkey=dlbyhkpxo7ptq47k9a3sme3zz&dl=1",
    "https://www.dropbox.com/scl/fi/cok57cd7olp88sa6iqagz/GraspNet_S6_1_Source_data.zip?rlkey=awh4tqu9ulezm8h5nmzrvo6ez&dl=1",
    "https://www.dropbox.com/scl/fi/xsdgoc65cinmekuq4emy4/GraspNet_S7_1_Source_data.zip?rlkey=5xvya2f4qz877t0eqwx11653l&dl=1",
    "https://www.dropbox.com/scl/fi/kzbdc6y9wbjqtwegxt6ub/GraspNet_S8_1_Source_data.zip?rlkey=ossp04yg1pip6fn4j1we5e06c&dl=1",
    "https://www.dropbox.com/scl/fi/imnr1ks8ivb3csmvq9189/GraspNet_S9_1_Source_data.zip?rlkey=1aq25mm7ofudvt72795x4na1s&dl=1",
    "https://www.dropbox.com/scl/fi/dv6zbs1lrd85d5ih0rqqk/GraspNet_S10_1_Source_data.zip?rlkey=c1hxbpp8mtaon9vorfdyknhn1&dl=1",
    "https://www.dropbox.com/scl/fi/eptfmx0eardrwjnyecd54/GraspNet_S11_1_Source_data.zip?rlkey=xik0pux9eyurb4yi0kt2t4oza&dl=1",
    "https://www.dropbox.com/scl/fi/qcjmq765tf2yeoaswze6m/GraspNet_S12_1_Source_data.zip?rlkey=zzufgp078law40qcyjlataeuz&dl=1",
    "https://www.dropbox.com/scl/fi/ko2uojypaabxkjbra5p0k/GraspNet_S13_1_Source_data.zip?rlkey=u71mu6pl8j6cnbpq2zr7x3hyp&dl=1",
    "https://www.dropbox.com/scl/fi/scvfht72w4v8il4y3wq46/GraspNet_S14_1_Source_data.zip?rlkey=tt5pg77t307tmhz6u601ccas4&dl=1",
    "https://www.dropbox.com/scl/fi/kqx3jf9bcttwz3k9xkp8z/GraspNet_S15_1_Source_data.zip?rlkey=zcwhqk25wqho4oocqdyxqpmxy&dl=1",
    "https://www.dropbox.com/scl/fi/15kc4z0cimgn8vlb1nimt/GraspNet_S16_1_Source_data.zip?rlkey=tm2gju9ypme1yqnnkiuwbckw0&dl=1",
    "https://www.dropbox.com/scl/fi/iy6q8mnwtchqsvht1zh9x/GraspNet_S17_1_Source_data.zip?rlkey=nvp0nvmmqb3e9dc73ggbql15j&dl=1",
    "https://www.dropbox.com/scl/fi/wpwln5fj3q0n8qwr58ucx/GraspNet_S18_1_Source_data.zip?rlkey=thg4j71bizs1i5fthz77kqx4c&dl=1",
    "https://www.dropbox.com/scl/fi/k04xlukf0t8v1jv8pb8j5/GraspNet_S19_1_Source_data.zip?rlkey=t6ao3gv0n1zoxo4r7195y2f8y&dl=1",
    "https://www.dropbox.com/scl/fi/6tnpc6zot73rtpq4bmceh/GraspNet_S20_1_Source_data.zip?rlkey=onj2xlwftiutu26pefvphq0lh&dl=1",
    "https://www.dropbox.com/scl/fi/owuihethwpdgpyhnjcdcg/GraspNet_S21_1_Source_data.zip?rlkey=pelrugt27u9zbyr38rmzy2jr1&dl=1",
    "https://www.dropbox.com/scl/fi/ra8tte29y0v4j87av9c81/GraspNet_S22_1_Source_data.zip?rlkey=pxn5mw66lq4agy76xg7ofaq06&dl=1",
    "https://www.dropbox.com/scl/fi/217j61ahnh3rvuqbp4r4t/GraspNet_S23_1_Source_data.zip?rlkey=4ef1trgi44mljj48c73zitl76&dl=1",
    "https://www.dropbox.com/scl/fi/rc3gtttmincxefgjmlrg4/GraspNet_S24_1_Source_data.zip?rlkey=yhclr5oc0sq3k7q75i7f1rjbn&dl=1",
    "https://www.dropbox.com/scl/fi/kuq3hn1gem8v8ofvoh7wd/GraspNet_S25_1_Source_data.zip?rlkey=e1efiadfm9kjga86sgpg2wb2p&dl=1",
    "https://www.dropbox.com/scl/fi/3he1zo3eddqoiugqve3s4/GraspNet_S26_1_Source_data.zip?rlkey=yue9qy9jh3aqy2z5n1dz3kgxo&dl=1",
    "https://www.dropbox.com/scl/fi/dasbdozeqjc98lxpkkcme/GraspNet_S27_1_Source_data.zip?rlkey=drz78ufosaaxm0kpm3z0zdfcp&dl=1",
    "https://www.dropbox.com/scl/fi/3czenftir31q1twzo1dwd/GraspNet_S28_1_Source_data.zip?rlkey=q10vx1ik9ty70l9g4mdq803zy&dl=1",
    "https://www.dropbox.com/scl/fi/txa2bqin3x8vr67mxy6cw/GraspNet_S29_1_Source_data.zip?rlkey=kwoj6tdn49auqic29jz3exe9c&dl=1",
    "https://www.dropbox.com/scl/fi/e2b7zwxxt7eacggkkh52b/GraspNet_S30_1_Source_data.zip?rlkey=dvsbeai8h8de7k62rh8wq036m&dl=1",
    "https://www.dropbox.com/scl/fi/tlgxafamzyvluecowzne9/GraspNet_S31_1_Source_data.zip?rlkey=pesq650mor4k8c5xez590rhfr&dl=1",
    "https://www.dropbox.com/scl/fi/isrpc304yz5hrnmw93h71/GraspNet_S32_1_Source_data.zip?rlkey=m7imhjjishjvvgmjgw7wpjnzx&dl=1",
    "https://www.dropbox.com/scl/fi/b1jslpzldd1uhmzwwz81z/GraspNet_S33_1_Source_data.zip?rlkey=xj8pmr2cazcojdyybbby4l6sb&dl=1",
    "https://www.dropbox.com/scl/fi/peampb49x03t6c801m4sa/GraspNet_S34_1_Source_data.zip?rlkey=zjsqxm2cvw1srtgsmg1m3r71p&dl=1",
    "https://www.dropbox.com/scl/fi/q0ig20fefcgi32al6ea4p/GraspNet_S35_1_Source_data.zip?rlkey=y9aohv8edfrzckxsf8wp4ei2p&dl=1",
    "https://www.dropbox.com/scl/fi/8gthpnsu0bqnx3pqyp5wo/GraspNet_S36_1_Source_data.zip?rlkey=z6enl62lfw2ie6iyp7rg6kmjw&dl=1",
    "https://www.dropbox.com/scl/fi/bbl952u6pqu3h87mqkvql/GraspNet_S37_1_Source_data.zip?rlkey=sktsco24vknwfd21gffngi5wd&dl=1",
    "https://www.dropbox.com/scl/fi/6omdgiyqqi76ct81uhph3/GraspNet_S38_1_Source_data.zip?rlkey=6jygxol362aha9q3gpjdbivtx&dl=1",
    "https://www.dropbox.com/scl/fi/navuh28us6gwsxmp5z57j/GraspNet_S39_1_Source_data.zip?rlkey=hw514wbgdgfetrkllbypzt2hg&dl=1",
    "https://www.dropbox.com/scl/fi/sud9v7bsbaxgvayfo2gdp/GraspNet_S40_1_Source_data.zip?rlkey=sp23gsu0vsxdhgzkwht7waszt&dl=1",
    "https://www.dropbox.com/scl/fi/46vwy7yc2b6oyt77svyyc/GraspNet_S41_1_Source_data.zip?rlkey=3op5zhn8yykna33qr25tucqdj&dl=1",
    "https://www.dropbox.com/scl/fi/1adxuj8vgnlxzgdkhp08p/GraspNet_S42_1_Source_data.zip?rlkey=noeiryu0h0oyzv6rz3ovgxeuy&dl=1",
    "https://www.dropbox.com/scl/fi/xcrvukabjwfquylrficwz/GraspNet_S43_1_Source_data.zip?rlkey=879640ebzyc3n94sm3cagusti&dl=1",
    "https://www.dropbox.com/scl/fi/82s27mlecnqzdvevuvpj5/GraspNet_S44_1_Source_data.zip?rlkey=rv270bfcwza1y7khmmhl8e9bb&dl=1",
    "https://www.dropbox.com/scl/fi/khjh4241ijs7895jdyw7t/GraspNet_S45_1_Source_data.zip?rlkey=gdesyupjcmaqcvhbfzitp7qre&dl=1",
    "https://www.dropbox.com/scl/fi/darp11j7fmchxk6sstf5z/GraspNet_S46_1_Source_data.zip?rlkey=rev6t96voq1yfgk1ynkxh3wso&dl=1",
    "https://www.dropbox.com/scl/fi/pzfn3k84h4r0pcj3yf61k/GraspNet_S47_1_Source_data.zip?rlkey=zsrpnw5chs44pfoo5p73cf57u&dl=1",
    "https://www.dropbox.com/scl/fi/ak21kbdnfwp3gxdu7tzzn/GraspNet_S48_1_Source_data.zip?rlkey=7lmdcdaao3infsnssqk0aik2n&dl=1",
    "https://www.dropbox.com/scl/fi/jpfsdmdu6jpsegjksf137/GraspNet_S49_1_Source_data.zip?rlkey=9fy1mcz7nbtjr2sc2zizh02j3&dl=1",
    "https://www.dropbox.com/scl/fi/jzgtzrut17f0e6k2veuy5/GraspNet_S50_1_Source_data.zip?rlkey=ae672s8pplhrdplotlxltaqei&dl=1",
    "https://www.dropbox.com/scl/fi/f559t78pfnlkkxaenklij/GraspNet_S51_1_Source_data.zip?rlkey=px2dykzmfbk77fjgzf0apfjcj&dl=1",
    "https://www.dropbox.com/scl/fi/201ih43c4afrr97r3lk5j/GraspNet_S52_1_Source_data.zip?rlkey=k8xqc70hidf2bzd08mc9gsdgv&dl=1",
    "https://www.dropbox.com/scl/fi/w69hhz0t75y6gka0epa71/GraspNet_S53_1_Source_data.zip?rlkey=yed6ur8xbls6bn98oyklrao68&dl=1",
    "https://www.dropbox.com/scl/fi/s1614t4g1fgn54luz28lj/GraspNet_S54_1_Source_data.zip?rlkey=7iduxe1lyt6oo6j8w8ossb6tr&dl=1",
    "https://www.dropbox.com/scl/fi/50bpkmtd1l5d9xjrsvukw/GraspNet_S55_1_Source_data.zip?rlkey=3z1zh0sfx8mqm51gietsywrxa&dl=1",
    "https://www.dropbox.com/scl/fi/a5i9yah9fkcanwmz0220m/GraspNet_S56_1_Source_data.zip?rlkey=6ve6mm45eu8wj43n4hbcmwrml&dl=1",
    "https://www.dropbox.com/scl/fi/kaop4lu76at47k5ym0g0y/GraspNet_S57_1_Source_data.zip?rlkey=k5aij089pdk4j0zd5sn474gm5&dl=1",
    "https://www.dropbox.com/scl/fi/vtx675m5yxw9oznudhlpn/GraspNet_S58_1_Source_data.zip?rlkey=67nng8rixooxki1yrbf41q93g&dl=1",
    "https://www.dropbox.com/scl/fi/rmbvwyz9345lofk41blr7/GraspNet_S59_1_Source_data.zip?rlkey=2ekadxzhr4w3smq2jnwfot4eu&dl=1",
    "https://www.dropbox.com/scl/fi/tdc2l3ae0nz3tckuwf2ha/GraspNet_S60_1_Source_data.zip?rlkey=xw0dhfoh6d5l2r6i53bl2ixd7&dl=1",
    "https://www.dropbox.com/scl/fi/1ln8bph76hewxtmrmwpt4/GraspNet_S61_1_Source_data.zip?rlkey=j17dsr31l0c6ka13clqot3y42&dl=1",
    "https://www.dropbox.com/scl/fi/63vztus5o69kcamc6f8h5/GraspNet_S62_1_Source_data.zip?rlkey=8n0lz7imqmmjbz2agxsf26vvt&dl=1",
    "https://www.dropbox.com/scl/fi/pco9iflyqdpwcbgr6ubb9/GraspNet_S63_1_Source_data.zip?rlkey=u413qurxlhqbaak0zko7e2mes&dl=1",
    "https://www.dropbox.com/scl/fi/ocdzg8ptvdnq5ic4xosiv/GraspNet_S64_1_Source_data.zip?rlkey=g1mkwz9zoly2yyh4fsttt0wpx&dl=1",
    "https://www.dropbox.com/scl/fi/lp0ixut9t8w55r3q52ude/GraspNet_S65_1_Source_data.zip?rlkey=n7ohvcdfnfeaddxlaimgbk3a7&dl=1",
    "https://www.dropbox.com/scl/fi/von3cks1t35s9zl9kqih0/GraspNet_S66_1_Source_data.zip?rlkey=8h8b2kzi6ogq81pe1svze9ah7&dl=1",
    "https://www.dropbox.com/scl/fi/4fnkqowk74d52zk54nvt0/GraspNet_S67_1_Source_data.zip?rlkey=q2qege9mybubsezlhatb4oe9b&dl=1",
    "https://www.dropbox.com/scl/fi/drcjmapmufcvaroh753wv/GraspNet_S68_1_Source_data.zip?rlkey=8jylv4dgbx0obu8m713j7wjka&dl=1",
    "https://www.dropbox.com/scl/fi/0htontuvv5ilb8ddn95lh/GraspNet_S69_1_Source_data.zip?rlkey=z84wj115g5zc06a0kkaou0qbs&dl=1",
    "https://www.dropbox.com/scl/fi/16auws6d2c7lalrc850yr/GraspNet_S70_1_Source_data.zip?rlkey=jh7dcjnilh6j2ghjjb9ix2at0&dl=1",
    "https://www.dropbox.com/scl/fi/z9prvnf41ixn99zpaipzb/GraspNet_S71_1_Source_data.zip?rlkey=subaa2ze3ztwo9jf1t5d88g9x&dl=1",
    "https://www.dropbox.com/scl/fi/q4xggvxpumx6o8ohb5hpy/GraspNet_S72_1_Source_data.zip?rlkey=3x5tnjy4paxfme37ozhojg7v7&dl=1",
    "https://www.dropbox.com/scl/fi/kx9avgb968wimyb9xadhp/GraspNet_S73_1_Source_data.zip?rlkey=0d5p0a1x7wqm55hqr81rehdfw&dl=1",
    "https://www.dropbox.com/scl/fi/gd9r718f8glwaomm9clgc/GraspNet_S74_1_Source_data.zip?rlkey=d9qgnghdfsdy7al30eb7jotil&dl=1",
    "https://www.dropbox.com/scl/fi/nbjufvf4wobrtgdsrp9ip/GraspNet_S75_1_Source_data.zip?rlkey=tevb8b7bodbh2ujxaskoyhqzo&dl=1",
    "https://www.dropbox.com/scl/fi/loqyt2ppt6mfdqpilwdw7/GraspNet_S76_1_Source_data.zip?rlkey=jjnv5hk12xvm402baki7niynf&dl=1",
    "https://www.dropbox.com/scl/fi/d34ci4ovyozl8sz05an65/GraspNet_S77_1_Source_data.zip?rlkey=z6xxu183udiwdusqk9tpvn29r&dl=1",
    "https://www.dropbox.com/scl/fi/x0scxwo0zg4tqh1ab9wju/GraspNet_S78_1_Source_data.zip?rlkey=b8ngsoupj2o1epwxyuyemvka0&dl=1",
    "https://www.dropbox.com/scl/fi/sw51urjomg1moldzmkuuz/GraspNet_S79_1_Source_data.zip?rlkey=ksuzdcc4mrv02lj90d1ho1bk7&dl=1",
    "https://www.dropbox.com/scl/fi/f7rxpwzigryicpx1li2zq/GraspNet_S80_1_Source_data.zip?rlkey=0ndms22vnjkgy2y0cjicytqef&dl=1",
    "https://www.dropbox.com/scl/fi/hi7u86v8r4cwth8wrnlda/GraspNet_S81_1_Source_data.zip?rlkey=icz1u5ulqo7n9ab7qogc89xzp&dl=1",
    "https://www.dropbox.com/scl/fi/n2473lp9uyjef366mnl8w/GraspNet_S82_1_Source_data.zip?rlkey=sycc3d3qqcypxps9ryqw4y4rp&dl=1",
    "https://www.dropbox.com/scl/fi/p78w8maf9oo1i0fjh483r/GraspNet_S83_1_Source_data.zip?rlkey=4dxvormejql176b74ut7n6hda&dl=1",
    "https://www.dropbox.com/scl/fi/d7itae6vy60761jj7c01r/GraspNet_S84_1_Source_data.zip?rlkey=pxer8phlpstgx2kvf5mfvo7io&dl=1",
    "https://www.dropbox.com/scl/fi/g2drsl9mw5nh4jur1ttiw/GraspNet_S85_1_Source_data.zip?rlkey=vwrn7eu1apbrcebhes50d7gt5&dl=1",
    "https://www.dropbox.com/scl/fi/0eoh1b7j2y9lrjzcx7ruj/GraspNet_S86_1_Source_data.zip?rlkey=lxp1u63fsp7iy6pspswh3ddfx&dl=1",
    "https://www.dropbox.com/scl/fi/zf9xpnq5pansdeyju2q9h/GraspNet_S87_1_Source_data.zip?rlkey=kzu4cixwy7fswzkvqqn8xinhv&dl=1",
    "https://www.dropbox.com/scl/fi/allwpijqk9wiek53ltdrr/GraspNet_S88_1_Source_data.zip?rlkey=fmikd1thdc10nzlgw5g2m2dpb&dl=1",
    "https://www.dropbox.com/scl/fi/xfms0i0rf47nz8dk5gqjo/GraspNet_S89_1_Source_data.zip?rlkey=n17ene1t1wg1oibqklzfa9k5h&dl=1",
    "https://www.dropbox.com/scl/fi/3r8z7nv3zfyt1musikrl4/GraspNet_S90_1_Source_data.zip?rlkey=ofh4sa0cgfr1kounhwgfonfx4&dl=1",
    "https://www.dropbox.com/scl/fi/6h6ahn9xdgdo14nlh7wod/GraspNet_S91_1_Source_data.zip?rlkey=y0ozvr3tskube2hem01pscelc&dl=1",
    "https://www.dropbox.com/scl/fi/n19m53z373gupeyz5bf9t/GraspNet_S92_1_Source_data.zip?rlkey=t8sn9qog5czh3s8ek7gsr9o48&dl=1",
    "https://www.dropbox.com/scl/fi/5bf64tpfq51xgd5ox3plj/GraspNet_S93_1_Source_data.zip?rlkey=bwmhart71h97jini3azlgocjo&dl=1",
    "https://www.dropbox.com/scl/fi/eigc3c8qbo3x3twawzmdb/GraspNet_S94_1_Source_data.zip?rlkey=6px73zxa45x1u85sqlfj81ff5&dl=1",
    "https://www.dropbox.com/scl/fi/ma6bmj8fcaihzp9xwlnh7/GraspNet_S95_1_Source_data.zip?rlkey=dw5n0cd93zoqk8537q35qk4z1&dl=1",
    "https://www.dropbox.com/scl/fi/hg2riwhvykbbmcspm6i9k/GraspNet_S96_1_Source_data.zip?rlkey=1qai2hyxmu1m21tjt191tcsk9&dl=1",
    "https://www.dropbox.com/scl/fi/yirunais37m4g9wpqo5ux/GraspNet_S97_1_Source_data.zip?rlkey=tyaxm1i8wzm01vgkc5by0uznw&dl=1",
    "https://www.dropbox.com/scl/fi/0epxalfc3mjcx1o8vu91s/GraspNet_S98_1_Source_data.zip?rlkey=wtjsyax1ow2tvbk8j2lh0m206&dl=1",
    "https://www.dropbox.com/scl/fi/lvy394b559220boreruxd/GraspNet_S99_1_Source_data.zip?rlkey=2aao3o2obv5jey3cjb576pg3z&dl=1",
]

print(f"CSV entrada: {CSV_IN}")
print(f"CSV salida:  {CSV_OUT}")

In [ ]:
# Celda 4: Cargar frame index (sin columnas MANO para ahorrar RAM)
import pandas as pd

MANO_COLS = [f'MANO_pose_{i:02d}' for i in range(45)]
all_cols  = pd.read_csv(CSV_IN, nrows=0).columns.tolist()
load_cols = [c for c in all_cols if c not in MANO_COLS]

print(f"Cargando frame index ({len(load_cols)} columnas, sin MANO pose)...")
df_all = pd.read_csv(CSV_IN, usecols=load_cols)
print(f"Total frames: {len(df_all):,} | Columnas: {len(df_all.columns)}")
print(f"Sujetos: {sorted(df_all['subject_id'].unique())}")

In [ ]:
# Celda 5: Definir pipeline MediaPipe + HaMeR
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
import time, urllib.request, os, zipfile, shutil, csv, subprocess

CROP_SIZE     = 256
PADDING       = 0.3
WRIST_IDX     = 0
INDEX_MCP_IDX = 5
TARGET_DIST   = 0.1

JOINTS = [
    'WRIST',
    'THUMB_CMC', 'THUMB_MCP', 'THUMB_IP', 'THUMB_TIP',
    'INDEX_FINGER_MCP', 'INDEX_FINGER_PIP', 'INDEX_FINGER_DIP', 'INDEX_FINGER_TIP',
    'MIDDLE_FINGER_MCP', 'MIDDLE_FINGER_PIP', 'MIDDLE_FINGER_DIP', 'MIDDLE_FINGER_TIP',
    'RING_FINGER_MCP', 'RING_FINGER_PIP', 'RING_FINGER_DIP', 'RING_FINGER_TIP',
    'PINKY_MCP', 'PINKY_PIP', 'PINKY_DIP', 'PINKY_TIP',
]
XYZ_COLS  = [f'{j}_{ax}' for j in JOINTS for ax in ('x', 'y', 'z')]
MANO_COLS = [f'MANO_pose_{i:02d}' for i in range(45)]
OUT_COLS  = load_cols + MANO_COLS  # XYZ ya estan en load_cols (se sobreescriben)

# Descargar modelo MediaPipe
MODEL_PATH = "/content/hand_landmarker.task"
if not os.path.exists(MODEL_PATH):
    print("Descargando modelo MediaPipe...")
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task",
        MODEL_PATH
    )

base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
options = mp_vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,
    min_hand_detection_confidence=0.3,
    min_hand_presence_confidence=0.3,
    min_tracking_confidence=0.3,
    running_mode=mp_vision.RunningMode.IMAGE,
)
detector = mp_vision.HandLandmarker.create_from_options(options)

def normalize(pts):
    pts = pts.copy()
    pts -= pts[WRIST_IDX]
    d = np.linalg.norm(pts[INDEX_MCP_IDX])
    if d > 1e-6:
        pts *= TARGET_DIST / d
    return pts

def run_pipeline(img_path):
    frame = cv2.imread(str(img_path))
    if frame is None:
        return None, None, "read_error"
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = detector.detect(mp_image)
    if not result.hand_landmarks:
        return None, None, "no_hand"
    if result.handedness:
        label    = result.handedness[0][0].category_name
        is_right = (label == "Right")
    else:
        is_right = True
    lms = result.hand_landmarks[0]
    xs  = [lm.x for lm in lms]
    ys  = [lm.y for lm in lms]
    h, w = frame.shape[:2]
    x1 = max(0.0, min(xs) - PADDING)
    y1 = max(0.0, min(ys) - PADDING)
    x2 = min(1.0, max(xs) + PADDING)
    y2 = min(1.0, max(ys) + PADDING)
    crop = frame[int(y1*h):int(y2*h), int(x1*w):int(x2*w)]
    if crop.size == 0:
        return None, None, "crop_empty"
    crop = cv2.resize(crop, (CROP_SIZE, CROP_SIZE))
    hand_side = "Right" if is_right else "Left"
    keypoints, _ = hamer_model.process(crop, {hand_side: [0.0, 0.0, 1.0, 1.0]}, rescale_factor=1.0)
    if hand_side not in keypoints:
        return None, None, "hamer_fail"
    kp = np.array(keypoints[hand_side], dtype=np.float32)
    try:
        rotmats = hamer_model.result["pred_mano_hand_pose"][0].numpy()
        aa_list = []
        for rot in rotmats:
            aa, _ = cv2.Rodrigues(rot)
            aa_list.extend(aa.flatten().tolist())
    except Exception:
        return None, None, "pose_fail"
    kp_norm = normalize(kp)
    if not is_right:
        kp_norm[:, 0] *= -1.0
    return kp_norm, aa_list, None

def img_path_from_row(source_root, row):
    parts    = row['sequence_id'].split('/')
    seq_base = parts[0]
    trial    = parts[1]
    cam      = row['cam']
    fid      = int(row['frame_id'])
    return source_root / seq_base / trial / 'rgb' / cam / f'{cam}_{fid}.jpg'

def find_source_root(extract_dir):
    """Encuentra el directorio raiz que contiene las carpetas de secuencia."""
    for root, dirs, files in os.walk(str(extract_dir)):
        if 'rgb' in dirs:
            # Sube 2 niveles: rgb -> trial -> sequence -> source_root
            return Path(root).parent.parent
    # Fallback: buscar primer nivel con subdirectorios de fecha
    for d in sorted(extract_dir.iterdir()):
        if d.is_dir():
            return d.parent
    return extract_dir

print("Pipeline definido.")

In [ ]:
# Celda 6: Loop principal -- S2 a S99

# Cargar done_keys del CSV existente
done_keys = set()
if CSV_OUT.exists():
    done_df = pd.read_csv(CSV_OUT, usecols=['subject_id', 'sequence_id', 'cam', 'frame_id'])
    for _, r in done_df.iterrows():
        done_keys.add((int(r['subject_id']), r['sequence_id'], r['cam'], int(r['frame_id'])))
    print(f"Frames ya procesados: {len(done_keys):,}")

# Cargar sujetos completamente terminados
done_subjects = set()
if PROGRESS_FILE.exists():
    done_subjects = set(int(x) for x in PROGRESS_FILE.read_text().split() if x.strip())
    print(f"Sujetos completos: {sorted(done_subjects)}")

write_header = not CSV_OUT.exists()

failed_log = open(LOG_FAILED, 'a')

with open(CSV_OUT, 'a', newline='') as f_out:
    writer = csv.DictWriter(f_out, fieldnames=OUT_COLS)
    if write_header:
        writer.writeheader()

    for subj_id in range(START_SUBJECT, END_SUBJECT + 1):
        if subj_id in done_subjects:
            print(f"[S{subj_id:02d}] Ya completado, saltando.")
            continue

        df_subj = df_all[df_all['subject_id'] == subj_id].reset_index(drop=True)
        if len(df_subj) == 0:
            print(f"[S{subj_id:02d}] No hay frames en el CSV, saltando.")
            continue

        pending = [(i, row) for i, row in df_subj.iterrows()
                   if (subj_id, row['sequence_id'], row['cam'], int(row['frame_id'])) not in done_keys]

        if len(pending) == 0:
            print(f"[S{subj_id:02d}] Todos los frames ya en CSV, marcando completo.")
            done_subjects.add(subj_id)
            with open(PROGRESS_FILE, 'a') as pf:
                pf.write(f"{subj_id}\n")
            continue

        print(f"\n{'='*60}")
        print(f"[S{subj_id:02d}] {len(df_subj):,} frames totales | {len(pending):,} pendientes")

        # Descargar zip
        url      = SOURCE_URLS[subj_id - 1]
        zip_path = CONTENT_DIR / f"GraspNet_S{subj_id:02d}_source.zip"
        extract  = CONTENT_DIR / f"S{subj_id:02d}_frames"

        if not zip_path.exists():
            print(f"[S{subj_id:02d}] Descargando zip...")
            subprocess.run(
                ["wget", "-q", "--show-progress", "-O", str(zip_path), url],
                check=True
            )
            print(f"[S{subj_id:02d}] Descarga OK ({zip_path.stat().st_size / 1e9:.1f} GB)")

        if not extract.exists():
            print(f"[S{subj_id:02d}] Extrayendo...")
            extract.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_path, 'r') as zf:
                zf.extractall(extract)
            print(f"[S{subj_id:02d}] Extraccion OK")

        source_root = find_source_root(extract)
        print(f"[S{subj_id:02d}] source_root: {source_root}")

        # Procesar frames pendientes
        n_ok = n_fail = 0
        t_subj = time.time()

        for idx, (_, row) in enumerate(pending):
            img = img_path_from_row(source_root, row)
            if not img.exists():
                n_fail += 1
                failed_log.write(f"missing|S{subj_id}|{img}\n")
                failed_log.flush()
                continue

            kp, pose, err = run_pipeline(img)

            if kp is None:
                n_fail += 1
                failed_log.write(f"{err}|S{subj_id}|{img}\n")
                failed_log.flush()
                continue

            out_row = row.to_dict()
            for j, joint in enumerate(JOINTS):
                out_row[f'{joint}_x'] = float(kp[j, 0])
                out_row[f'{joint}_y'] = float(kp[j, 1])
                out_row[f'{joint}_z'] = float(kp[j, 2])
            for k in range(45):
                out_row[f'MANO_pose_{k:02d}'] = float(pose[k])

            writer.writerow(out_row)
            f_out.flush()
            n_ok += 1

            if (idx + 1) % 100 == 0:
                elapsed = time.time() - t_subj
                fps     = (idx + 1) / elapsed
                eta_s   = (len(pending) - idx - 1) / fps if fps > 0 else 0
                print(f"  [S{subj_id:02d} {idx+1}/{len(pending)}] "
                      f"{fps:.2f} fps | ok={n_ok} fail={n_fail} | "
                      f"ETA {eta_s/60:.1f} min")

        elapsed_subj = time.time() - t_subj
        print(f"[S{subj_id:02d}] COMPLETO: ok={n_ok} fail={n_fail} | "
              f"{elapsed_subj/60:.1f} min | {(n_ok+n_fail)/elapsed_subj:.2f} fps")

        # Marcar sujeto como terminado
        done_subjects.add(subj_id)
        with open(PROGRESS_FILE, 'a') as pf:
            pf.write(f"{subj_id}\n")

        # Limpiar /content/
        zip_path.unlink(missing_ok=True)
        shutil.rmtree(extract, ignore_errors=True)
        print(f"[S{subj_id:02d}] Limpieza OK")

failed_log.close()
print("\n=== SESION TERMINADA ===")
print(f"Sujetos completos esta sesion: {sorted(done_subjects)}")
print(f"CSV: {CSV_OUT}")